# 09e — Esercitazione: Function Calling e MCP

**Corso**: Programmazione di Applicazioni Intelligenti  
**Lezione 09** — Dal Client OpenAI al Function Calling e MCP  
**Blocco 4** — Esercitazione (1h)

In questa esercitazione metterai in pratica:
- Definire tool con JSON Schema e implementare il loop di function calling
- Gestire più tool nella stessa conversazione
- Creare un MCP Server con FastMCP
- (Bonus) Rendere sicuro il tool `calculate`

Ogni esercizio ha una **traccia con `# TODO`** da completare. Le soluzioni sono nel notebook `09f`.


---
## Setup


In [ ]:
!pip install -q openai

from openai import OpenAI
from google.colab import userdata
import json

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY"),
)
MODEL = "openai/gpt-oss-120b"
print(f"Client pronto! Modello: {MODEL}")


---
## Esercizio 1 — Function calling base (15 min)

### Obiettivo
Implementare il flusso completo di function calling con un singolo tool: `get_exchange_rate`.

### Consegna
1. Definisci il tool `get_exchange_rate` con JSON Schema:
   - Parametri: `from_currency` (str, required), `to_currency` (str, required)
   - Descrizione chiara per il modello
2. Implementa la funzione Python (dati fake, es. un dizionario di tassi di cambio)
3. Crea il registry
4. Scrivi il loop completo: chiamata → check tool_calls → esecuzione → seconda chiamata
5. Testa con: "Quanto vale 100 euro in dollari?"

### Nota
Il loop che scrivi qui gestisce un **singolo round** di tool calling: il modello può chiamare
uno o più tool in parallelo, ma dopo la risposta finale il loop termina. Nelle prossime lezioni
vedremo il pattern **ReAct**, che ripete il loop autonomamente fino a completare il compito.


In [ ]:
# Esercizio 1 — Definizione del tool

# TODO: definisci il tool get_exchange_rate come JSON Schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "TODO",  # TODO: descrizione chiara per il modello
            "parameters": {
                "type": "object",
                "properties": {
                    # TODO: definisci i parametri from_currency e to_currency
                },
                "required": []  # TODO: quali parametri sono obbligatori?
            }
        }
    }
]


In [ ]:
# Esercizio 1 — Funzione Python e registry

def get_exchange_rate(from_currency: str, to_currency: str) -> str:
    """Restituisce il tasso di cambio (dati simulati)."""
    # TODO: implementa con un dizionario di tassi fake
    # es. {"EUR_USD": 1.08, "USD_EUR": 0.93, "EUR_GBP": 0.86, ...}
    rates = {
        # TODO: aggiungi almeno 4 coppie di valute
    }

    key = f"{from_currency.upper()}_{to_currency.upper()}"
    # TODO: cerca il tasso nel dizionario
    # Se non trovato, restituisci un messaggio di errore
    # Restituisci JSON con json.dumps()
    pass  # TODO: sostituisci con il return


# Registry: mappa nome_tool -> funzione Python
tool_registry = {
    # TODO: "get_exchange_rate": ...
}


In [ ]:
# Esercizio 1 — Loop completo

def ask_with_tools(question: str) -> str:
    """Gestisce una domanda con function calling (singolo round)."""
    messages = [
        {"role": "system", "content": "Sei un assistente finanziario. Usa i tool disponibili. Rispondi in italiano."},
        {"role": "user", "content": question}
    ]

    # TODO: prima chiamata con tools
    response = None  # TODO: client.chat.completions.create(...)

    message = response.choices[0].message

    # Se il modello non chiede tool, restituisci direttamente
    if not message.tool_calls:
        return message.content

    # Aggiungi il messaggio dell'assistente (con le tool_calls) alla cronologia
    messages.append(message)

    # Per ogni tool_call, esegui la funzione e aggiungi il risultato
    for tool_call in message.tool_calls:
        fn_name = tool_call.function.name
        fn_args = json.loads(tool_call.function.arguments)

        # TODO: esegui la funzione dal registry
        result = None  # TODO: tool_registry[fn_name](**fn_args)

        # TODO: aggiungi il risultato con role "tool"
        messages.append({
            "role": "tool",
            # TODO: servono anche "tool_call_id" e "content"
        })

    # TODO: seconda chiamata per la risposta finale
    final_response = None  # TODO: client.chat.completions.create(...)
    return final_response.choices[0].message.content

# Test
print(ask_with_tools("Quanto vale 100 euro in dollari?"))


---
## Esercizio 2 — Multi-tool (15 min)

### Obiettivo
Aggiungere un secondo tool (`translate_text`) e testare come il modello gestisce **domande che richiedono più tool**.

### Consegna
1. Aggiungi alla lista `tools` un nuovo tool `translate_text`:
   - Parametri: `text` (str), `target_language` (str)
   - Simula la traduzione con un dizionario o restituendo il testo con un prefisso
2. Implementa la funzione Python e aggiungila al registry
3. Testa con domande che richiedono:
   - Solo il tool cambio valuta
   - Solo il tool traduzione
   - Entrambi (es. "Traduci in inglese: il tasso EUR/USD è...")
4. Osserva: il modello chiama entrambi i tool in parallelo o ne chiama uno e poi l'altro?

### Nota
Puoi riutilizzare la funzione `ask_with_tools` dell'esercizio 1 — il loop gestisce già tool paralleli.
Però il nostro loop fa un **singolo round**: se il modello ha bisogno del risultato del primo tool
per formulare la chiamata al secondo (dipendenza sequenziale), il secondo tool non verrà chiamato.
Rifletti: in quali casi questo è un limite?


In [ ]:
# Esercizio 2 — Aggiungi il tool translate_text

# TODO: aggiungi il tool translate_text alla lista tools
tools.append({
    "type": "function",
    "function": {
        "name": "translate_text",
        "description": "TODO",  # TODO: descrizione per il modello
        "parameters": {
            # TODO: definisci "text" e "target_language"
        }
    }
})

# TODO: implementa la funzione translate_text
def translate_text(text: str, target_language: str) -> str:
    # TODO: simulazione di traduzione
    # Suggerimento: puoi usare un dizionario di traduzioni semplici
    # o restituire f"[{target_language}] {text}" come placeholder
    pass  # TODO

# TODO: aggiungi al registry
tool_registry["translate_text"] = translate_text


In [ ]:
# Esercizio 2 — Test con domande diverse

# Test A: solo cambio valuta
print("=== Test A: solo cambio ===")
print(ask_with_tools("Qual è il tasso di cambio euro-sterlina?"))
print()

# Test B: solo traduzione
print("=== Test B: solo traduzione ===")
print(ask_with_tools("Traduci 'buongiorno' in inglese"))
print()

# Test C: domanda che potrebbe richiedere entrambi
# Nota: il modello potrebbe chiamare entrambi i tool in parallelo,
# oppure solo uno e poi completare da solo. Osserva il comportamento!
print("=== Test C: entrambi? ===")
print(ask_with_tools("Quanto vale 1 euro in dollari? Traduci la risposta in inglese."))


---
## Esercizio 3 — MCP Server (20 min)

### Obiettivo
Creare un **MCP Server** con FastMCP che gestisce una lista di task (todo list).

### Consegna
1. Completa il codice del server con tre tool:
   - `add_task(title: str, priority: str = "medium")` → aggiunge un task
   - `list_tasks()` → elenca tutti i task
   - `complete_task(task_id: int)` → segna un task come completato
2. Lo stato è in-memory (una lista Python)
3. **Test in Colab**: usa `mcp.list_tools()` e `mcp.call_tool()` per ispezionare il server
4. **Test in locale** (opzionale): lancia con MCP Inspector:
   ```
   pip install "mcp[cli]"
   npx @modelcontextprotocol/inspector python todo_server.py
   ```


In [ ]:
!pip install -q "mcp[cli]"


In [ ]:
# Esercizio 3 — MCP Server con FastMCP

from mcp.server.fastmcp import FastMCP

mcp_server = FastMCP("Todo Server")

# Stato in-memory
tasks = []
next_id = 1

@mcp_server.tool()
def add_task(title: str, priority: str = "medium") -> str:
    """Aggiunge un task alla lista."""
    # TODO: aggiungi il task alla lista
    # Ogni task ha: id, title, priority, completed (bool)
    # Ricorda di usare 'global next_id' per aggiornare il contatore
    # Restituisci una conferma come stringa
    pass  # TODO

@mcp_server.tool()
def list_tasks() -> str:
    """Elenca tutti i task con id, titolo, priorita' e stato."""
    # TODO: restituisci la lista dei task come stringa formattata
    # Gestisci il caso lista vuota
    pass  # TODO

@mcp_server.tool()
def complete_task(task_id: int) -> str:
    """Segna un task come completato dato il suo id."""
    # TODO: segna il task con l'id specificato come completato
    # Gestisci il caso in cui l'id non esiste
    pass  # TODO

print("Server MCP definito! Esegui la cella successiva per testarlo.")


In [ ]:
# Esercizio 3 — Test del server in Colab
# Non serve lanciare mcp_server.run(): ispezioniamo direttamente l'oggetto.

import asyncio

async def test_todo_server():
    # 1. Verifica i tool registrati
    print("=== Tool registrati ===")
    tools_list = await mcp_server.list_tools()
    for t in tools_list:
        print(f"  {t.name}: {t.description}")
        print(f"    Schema: {t.inputSchema}")
    print()

    # 2. Aggiungi qualche task
    print("=== Aggiunta task ===")
    r1 = await mcp_server.call_tool("add_task", {"title": "Studiare function calling", "priority": "high"})
    print(f"  {r1[0][0].text}")
    r2 = await mcp_server.call_tool("add_task", {"title": "Fare la spesa"})
    print(f"  {r2[0][0].text}")
    print()

    # 3. Lista task
    print("=== Lista task ===")
    r3 = await mcp_server.call_tool("list_tasks", {})
    print(f"  {r3[0][0].text}")
    print()

    # 4. Completa un task
    print("=== Completamento ===")
    r4 = await mcp_server.call_tool("complete_task", {"task_id": 1})
    print(f"  {r4[0][0].text}")
    print()

    # 5. Lista aggiornata
    print("=== Lista aggiornata ===")
    r5 = await mcp_server.call_tool("list_tasks", {})
    print(f"  {r5[0][0].text}")

await test_todo_server()


---
## Esercizio 4 (Bonus) — Guardrails di sicurezza (10 min)

### Obiettivo
Il tool `calculate` del Notebook 09d usa `eval()`, che è **pericoloso**: un utente malevolo potrebbe iniettare codice arbitrario.

### Consegna
1. Implementa una versione sicura `safe_calculate(expression)` che:
   - Accetta solo espressioni con numeri, operatori (`+`, `-`, `*`, `/`, `**`, `%`), parentesi e spazi
   - Rifiuta qualsiasi altra cosa (lettere, `import`, `__`, ecc.)
   - Usa un **regex di validazione** come primo filtro, poi `eval()` sull'espressione validata
2. Testa con input legittimi e malevoli:
   - `"15 * 23 + 7"` → deve funzionare
   - `"(100 + 50) / 3"` → deve funzionare
   - `"__import__('os').system('rm -rf /')"` → deve essere rifiutato
   - `"eval('print(1)')"` → deve essere rifiutato

### Nota
`ast.literal_eval` **non** funziona per le espressioni aritmetiche (supporta solo letterali
come stringhe, numeri e tuple). Il regex è l'approccio corretto qui.


In [ ]:
# Esercizio 4 (Bonus) — Versione sicura di calculate

import re

def safe_calculate(expression: str) -> str:
    # TODO: valida l'espressione con un regex
    # Solo: cifre, spazi, +, -, *, /, (, ), ., %, **
    # Suggerimento: il pattern deve accettare SOLO questi caratteri
    allowed_pattern = "TODO"  # TODO: scrivi il regex

    if not re.match(allowed_pattern, expression):
        return json.dumps({"error": f"Espressione non valida: {expression}"})

    # Se il pattern e' valido, calcola con eval()
    # (a questo punto e' ragionevolmente sicuro perche' abbiamo filtrato l'input)
    try:
        result = None  # TODO: eval(...)
        return json.dumps({"expression": expression, "result": result})
    except Exception as e:
        return json.dumps({"error": str(e)})


# Test con input legittimi
print("=== Input legittimi ===")
print(safe_calculate("15 * 23 + 7"))
print(safe_calculate("(100 + 50) / 3"))
print(safe_calculate("2 ** 10"))

# Test con input malevoli
print("\n=== Input malevoli (devono essere rifiutati) ===")
print(safe_calculate("__import__('os').system('rm -rf /')"))
print(safe_calculate("eval('print(1)')"))
print(safe_calculate("open('/etc/passwd').read()"))


---
## Fatto?

Confronta le tue soluzioni con il notebook delle soluzioni.

### Cosa abbiamo coperto in questa lezione:
- **Blocco 1-2**: la libreria `openai` come interfaccia universale (chat, streaming, reasoning, structured output)
- **Blocco 3-4**: function calling (il meccanismo) e MCP (il protocollo standard)

